# 02-First run：用 kmhv3.py 运行 hello

这一节尝试 GEM5 最基本的仿真流程：使用与 RTL 对齐的处理器配置 (`kmhv3.py`)，运行一个 hello RISC-V 程序。


## 运行 hello

In [ ]:
%%bash
set -euo pipefail
source ../env.sh

RUN="$WORK_DIR/04-gem5/02-first-run/precompiled"
rm -rf "$RUN/m5out"
mkdir -p "$RUN"

# 我们提供的 GEM5 模拟器 或 01 中编译好的 GEM5 模拟器
GEM5_BIN=$(get_asset gem5-precompile/gem5.fast)
# 即将运行的 hello 程序的 bin 文件
HELLO=$(get_asset workload/hello-riscv64-xs.bin)
# 让 GEM5 能够找到 DramSim3 库
export LD_LIBRARY_PATH="$(get_asset gem5-precompile):${LD_LIBRARY_PATH:-}"
# 与 GEM5 在线 difftest 的 nemu
export GCBV_REF_SO=$(get_asset gem5-precompile/riscv64-nemu-interpreter-so)

cd "$GEM5_HOME"

# 使用 --raw-cpt 参数来执行非切片程序，若是切片程序则不需要这个参数
# $GEM5_HOME/configs/example/kmhv3.py 是与 RTL kmhv3 精确对齐的处理器配置
"$GEM5_BIN" --outdir="$RUN/m5out" \
  "$GEM5_HOME/configs/example/kmhv3.py" \
  --raw-cpt --generic-rv-cpt="$HELLO" \
  2>&1 | tee "$RUN/run.log" >/dev/null

tr '\r' '\n' < "$RUN/run.log"

通过这一段 bash 脚本，XS-GEM5 仿真完成。运行结束之后可以输出 “Hello，Xiangshan!”

最后的 `m5_exit` 表示 guest 主动结束模拟。